In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_excel("../data/processed/ethiopia_fi_unified_data_enriched.xlsx")

# Target 1: Access
access = df[(df["indicator_code"] == "ACC_OWNERSHIP") & 
            (df["gender"] == "all") & (df["record_type"] == "observation")].copy()
access["year"] = pd.to_datetime(access["observation_date"]).dt.year
access = access.sort_values("year")[["year", "value_numeric"]]
print("Access (Account Ownership) — actual Findex points:")
print(access.to_string())

# Target 2: Usage — check what's actually available for "digital payment usage %"
usage_check = df[df["indicator_code"].isin(["USG_DIGITAL_PAYMENT", "USG_ACTIVE_RATE"])][
    ["record_id", "indicator", "indicator_code", "value_numeric", "observation_date"]
]
print("\nUsage-related % indicators found:")
print(usage_check.to_string())

Access (Account Ownership) — actual Findex points:
   year  value_numeric
0  2014           22.0
1  2017           35.0
2  2021           46.0
5  2024           49.0

Usage-related % indicators found:
   record_id                   indicator   indicator_code  value_numeric     observation_date
24  REC_0025  Mobile Money Activity Rate  USG_ACTIVE_RATE           66.0  2024-12-31 00:00:00


In [2]:
digital_search = df[df["indicator"].astype(str).str.contains("digital|payment", case=False, na=False)]
print(digital_search[["record_id", "indicator", "indicator_code", "value_numeric", "observation_date", "record_type"]].to_string())

   record_id                                 indicator     indicator_code  value_numeric     observation_date  record_type
11  REC_0012               Fayda Digital ID Enrollment          ACC_FAYDA      8000000.0  2024-08-15 00:00:00  observation
12  REC_0013               Fayda Digital ID Enrollment          ACC_FAYDA     12000000.0  2025-02-28 00:00:00  observation
13  REC_0014               Fayda Digital ID Enrollment          ACC_FAYDA     15000000.0  2025-05-15 00:00:00  observation
31  REC_0032               Fayda Digital ID Enrollment          ACC_FAYDA     90000000.0  2028-12-31 00:00:00       target
36  EVT_0004          Fayda Digital ID Program Rollout          EVT_FAYDA            NaN  2024-01-01 00:00:00        event
40  EVT_0008    EthioPay Instant Payment System Launch       EVT_ETHIOPAY            NaN  2025-12-18 00:00:00        event
62  EVT_0012  NBE Payment Instrument Issuers Directive  EVT_PII_DIRECTIVE            NaN           2020-04-01        event


In [4]:
new_usage_obs = pd.DataFrame([
    {
        "record_id": "REC_0062", "record_type": "observation", "pillar": "USAGE",
        "indicator": "Made or Received Digital Payment", "indicator_code": "USG_DIGITAL_PAYMENT",
        "value_numeric": 12.0, "gender": "all", "location": "national",
        "observation_date": "2017-12-31", "source_name": "World Bank Blog (Findex 2017 Ethiopia takeaways)",
        "source_type": "survey", "source_url": "https://blogs.worldbank.org/en/africacan/financial-inclusion-in-ethiopia-10-takeaways-from-findex-2017",
        "original_text": "Made or received digital payments in the past year (%) ... 12",
        "confidence": "high", "collected_by": "Maria", "collection_date": "2026-07-20",
        "notes": "Fills Usage target series gap for Task 4 forecasting; Ethiopia-specific Findex 2017 figure",
    },
    {
        "record_id": "REC_0063", "record_type": "observation", "pillar": "USAGE",
        "indicator": "Made or Received Digital Payment", "indicator_code": "USG_DIGITAL_PAYMENT",
        "value_numeric": 20.0, "gender": "all", "location": "national",
        "observation_date": "2021-12-31", "source_name": "World Bank Blog (Ethiopia mobile/account usage)",
        "source_type": "survey", "source_url": "https://blogs.worldbank.org/en/africacan/mobile-phone-technology-could-expand-equitable-access-financial-services-ethiopia",
        "original_text": "20% of adults\u2014used their accounts for digital payments",
        "confidence": "high", "collected_by": "Maria", "collection_date": "2026-07-20",
        "notes": "Fills Usage target series gap for Task 4 forecasting; Ethiopia-specific Findex 2021 figure",
    },
    {
        "record_id": "REC_0064", "record_type": "observation", "pillar": "USAGE",
        "indicator": "Made or Received Digital Payment", "indicator_code": "USG_DIGITAL_PAYMENT",
        "value_numeric": 35.0, "gender": "all", "location": "national",
        "observation_date": "2024-12-31", "source_name": "10Academy Week 11 Challenge Document",
        "source_type": "survey", "source_url": "internal_challenge_doc",
        "original_text": "Made or received digital payment: ~35%",
        "confidence": "medium", "collected_by": "Maria", "collection_date": "2026-07-20",
        "notes": "Approximate figure as stated in challenge doc overview; not independently re-verified against primary Findex 2024 source",
    },
])

df = pd.concat([df, new_usage_obs], ignore_index=True)
print(df.shape)
df.to_excel("../data/processed/ethiopia_fi_unified_data_enriched.xlsx", index=False)
print("Saved enriched dataset with USG_DIGITAL_PAYMENT series.")

(66, 35)
Saved enriched dataset with USG_DIGITAL_PAYMENT series.


## Modeling Approach

Given only 4 (Access) and 3 (Usage) historical points, a complex model
would overfit. Three complementary approaches, as suggested by the
challenge doc:

1. **Baseline trend regression** — simple linear regression on the
   Findex points alone. Captures the pure historical trajectory,
   ignoring events entirely. Serves as a naive benchmark.

2. **Event-augmented model** — baseline trend + the refined event-impact
   estimates from Task 3 (e.g., Telebirr's discounted +3pp effect,
   Fayda's partially-realized +10pp effect), projected forward using the
   same linear-ramp functional form.

3. **Scenario analysis** — optimistic/base/pessimistic bands built by
   varying the assumed strength of upcoming events (Fayda full rollout,
   EthioPay adoption, M-Pesa growth continuing) rather than just varying
   a statistical confidence interval, since with n=4 a purely statistical
   CI would be very wide and not very informative on its own.